In [1]:
from IPython.display import Image

- references
    - https://docs.pytorch.org/docs/stable/fsdp.html#torch.distributed.fsdp.CPUOffload
    - https://karthick.ai/blog/2024/Fully-Sharded-Data-Parallel-(FSDP)/
- torch.distributed.fsdp.CPUOffload(offload_params=False)
    - offload_params (bool) – This specifies whether to offload parameters to CPU when not involved in computation. If True, then this offloads gradients to CPU as well, meaning that the optimizer step runs on CPU.

- `python gpt2.py --gpus 6,7 --warmup 2 --iters 10`

```
==================== 测试场景: CPU Offload = True ====================
  [-] World Size: 2; 可见设备映射: 6,7
  [-] 初始底噪显存 (rank0): 0.00 MB
  [-] 创建模型 (默认在 CPU)...
  [-] FSDP 包装前显存 (rank0): 0.00 MB

  [-] FSDP 包装后静态显存 (跨 rank 最大值): 0.00 MB
  [-] rank0 参数设备: cpu
  [-] Forward 峰值显存 (跨 rank 最大值): 4082.86 MB
  [-] Forward 结束显存 (跨 rank 最大值): 2729.92 MB
  [-] 平均单步时延 (跨 rank 最大值): 59.40 ms
     (CPU offload 预期: 静态显存更低，但时延通常更高)
  ==================== 测试场景: CPU Offload = False ====================
  [-] World Size: 2; 可见设备映射: 6,7
  [-] 初始底噪显存 (rank0): 0.00 MB
  [-] 创建模型 (默认在 CPU)...
  [-] (常规模式) 手动将模型移动到 GPU
  [-] FSDP 包装前显存 (rank0): 2717.87 MB
  [-] FSDP 包装后静态显存 (跨 rank 最大值): 1364.94 MB
  [-] rank0 参数设备: cuda:0
  [-] Forward 峰值显存 (跨 rank 最大值): 4086.73 MB
  [-] Forward 结束显存 (跨 rank 最大值): 4082.86 MB
  [-] 平均单步时延 (跨 rank 最大值): 5.24 ms
```

### concepts

In [3]:
# https://huggingface.co/blog/Isayoften/optimization-rush#64-fsdp---fully-sharded-data-parallel
Image(url='https://cdn-uploads.huggingface.co/production/uploads/660710b03ef451aa2bab8971/7zvopsLHueGXc0FmKGi28.png', width=400)

- FSDP unit：被 FSDP(...) 包住的模块（可以是整模型，也可以是每个 block）
    - a logical grouping of model layers or sub-modules that are treated together for the purpose of sharding and communication during training.
- flat param：该 unit 的参数被拼成一个连续大张量
- local shard：flat param 按 world size 切分后，本 rank 持有的那一片
- unsharded/full param：计算时临时 all-gather 回来的完整参数
- FULL_SHARD：参数/梯度都做分片管理（ZeRO-3风格）
- cpu_offload=True：不参与计算时，把参数/梯度驻留到 CPU

### shard

- all-gather 的对象是当前 FSDP unit 的参数 shard（通常是一个被 FSDP 包裹的模块的 flat parameter）。
    - 一个 FSDP unit（你包的那个模块）里的参数会先被拍平成 flat param；
    - 然后这个 flat param 按 world size 切成不同 shards，每个 rank 常驻一片；
    - forward 前为当前 unit 临时 all-gather 成 full param，算完再 reshard（offload 开启则可回 CPU）。
- fsdp 的本质是 dp，每个 rank 都在跑完整模型的计算图，只是输入数据不同（各自的 mini-batch shard）。
- 每个 rank 不是“只算自己那部分 FSDP unit 的计算”，而是：
    - 对当前要执行的 FSDP unit，先 all-gather 出该 unit 的完整参数（临时）；
    - 然后用本 rank 的本地数据做这个 unit 的 forward/backward；
    - 结束后再把参数重新分片（reshard）。
- 反向时梯度通常走 reduce-scatter（而不是每卡保留完整梯度）。

### full shard vs. cpu offload

> offload 主要降低的是“常驻显存”
- 对于一个 FSDP unit（比如 layer2），它的参数被切成 N 份，每个 rank 都持有这个 unit 的一份 shard。
- 到了算 layer2 时，每个 rank 拿出自己手上的 layer2 shard，大家 all-gather 成 layer2 的 full param 来算本 rank 的数据。
- layer2 算完后，这个 full param 会被释放/reshard；若开了 offload，相关 shard 可回 CPU。

初始化/加载阶段（cpu_offload=True）

1. 模型先在 CPU 构建（你现在就是这样做的）
2. FSDP 对 unit 做 flatten，并按 rank 切分成 shard
3. 每个 rank 只保留自己的 local shard
4. 因为 cpu_offload=True，local shard 的长期驻留位置是 CPU（空闲时不占 GPU 常驻参数显存）

Forward 阶段（推理或训练的前向）对每个 FSDP unit，流程是：

1. 拿到 local shard（offload 场景下通常在 CPU）
2. 必要时做 CPU→GPU 传输
3. 各 rank 对该 unit 做 all-gather，临时拼成 full param
4. 用 full param 执行该 unit 的算子
5. unit forward 结束后，参数切回 sharded 视图；offload 场景下 shard 可回 CPU

Backward 阶段（训练时）训练会再多一轮类似过程：

1. 反向到某个 unit 时，再次需要该 unit full param（再 unshard / all-gather）
2. 计算梯度后，梯度通过 reduce-scatter 回到 shard 形态
3. cpu_offload=True 时，梯度也可驻留 CPU
4. 然后继续下一个 unit 的反向